# មេរៀន 16 - បង្ហោះភ្នាក់ងារដែលមានសមត្ថភាពធ្វើការ និង Microsoft Foundry

ក្នុងសៀវភៅកំណត់ត្រានេះ អ្នកបង្កើត **ភ្នាក់ងារជំនួយអតិថិជនដែលត្រៀមខ្លួនសម្រាប់ផលិតកម្ម** សម្រាប់ក្រុមហ៊ុននិទ្យស័ព្ទ **Contoso**។ ផ្ទុយពីមេរៀនមុនៗ គោលបំណងនៅទីនេះមិនមែនការប្រព្រឹត្តិចិត្តរបស់ភ្នាក់ងារនោះទេ — តែការចងជុំជុំវិញវាដែលធ្វើឱ្យភ្នាក់ងារមានសុវត្ថិភាពក្នុងការប្រតិបត្តិការscale:

1. **ការហៅប្រើឧបករណ៍** — ការស្វែងរកការបញ្ជាទិញ និងការបង្កើតសំបុត្រការងារ។
2. **RAG** — ចម្លើយគោលការណ៍ពីមូលដ្ឋានចំណេះដឹង។
3. **ការចងចាំ** — ទេពកោសល្យក្នុងការចងចាំអតិថិជនក្នុងបម្លាស់ប្ដូរពាក្យ។
4. **ការបញ្ជូនគំរូ** — ផ្ញើសំណើរងាយទៅកាន់គំរូតូចៗ ប៉ុន្តែសម្រាប់សំណើរលំបាកផ្ញើទៅគំរូធំ។
5. **ការបម្រុងទុកចម្លើយ** — បម្រុងទុកសំណួរដដែល ដោយមិនត្រូវហៅគំរូឡើងវិញ។
6. **ការអនុម័តពីមនុស្ស** — ការសងប្រាក់លើសដែនកំណត់ត្រូវបានពន្យារពេលរហូតដល់បានការអនុម័ត។
7. **ទ្វារវាយតម្លៃ** — សំណុំតេស្តពីក្រៅបណ្តាលឱ្យការចេញផ្សាយល្អាមានការកំណត់។
8. **ការអាចមើលឃើញ** — ការតាមដាន OpenTelemetry រួមជាមួយសំណើរៗគ្នាទាំងអស់។

មួយផ្នែកនិចជាឯករាជ្យ និងអាចដំណើរការបាន។ សូមអានម្តងមួយនៃបន្ទាត់ទាំងអស់ — ធាតុដើមផលិតកម្មត្រូវបានរក្សាឱ្យតូចបំផុត។


## ការតម្លើង

មុនពេលរត់សៀវភៅកំណត់ត្រានេះ សូមប្រាកដថាអ្នកមាន:

1. **គម្រោង Microsoft Foundry** ដែលមានគំរូជជែកបានដាក់បញ្ចូន (ឧ. `gpt-5-mini`)។
2. **បានចូលទៅក្នុង Azure CLI** — រត់ `az login` នៅក្នុងវែរគ្រងរបស់អ្នក។
3. **កំណត់អថេរបរិយាកាសដែលត្រូវការ៖**
   - `AZURE_AI_PROJECT_ENDPOINT` — ចំណុចផ្តល់សេវាគម្រោង Microsoft Foundry របស់អ្នក។
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះគំរូដែលបានដាក់បញ្ចូនរបស់អ្នក។

ផ្នែក RAG ប្រើប្រាស់ **Azure AI Search** នៅពេល `AZURE_SEARCH_SERVICE_ENDPOINT` និង `AZURE_SEARCH_API_KEY` ត្រូវបានកំណត់ ហើយបិទទៅរកការស្វែងរកក្នុងចិត្ត ដូច្នេះសៀវភៅកំណត់ត្រានេះរត់បានដោយគ្មានធនធានស្វែងរក។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. ឧបករណ៍

ឧបករណ៍ផលិតកម្មធ្វើការពិតប្រាកដលើប្រព័ន្ធពិតប្រាកដ។ នៅទីនេះយើងបង្ហាញអារាយការណ៍បញ្ជាទិញនិងប្រព័ន្ធសំបុត្រជាមួយមុខងារ Python ធម្មតា។ តំណក @tool បង្ហាញពួកវាទៅកាន់ភ្នាក់ងារ។

សូមយកចិត្តទុកដាក់ដែល `issue_refund` ប្រើ `approval_mode="always_require"` សម្រាប់ការវិលប្រាក់តាមលំដាប់លើទף — នេះជាសមាសភាគមនុស្សក្នុងស្វ័យប្រវត្តិនិងយើងនឹងប្រើបច្ចុប្បន្នហើយក្រោយមក។


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — មូលដ្ឋានចំណេះដឹងគោលការណ៍

សំណួរក្នុងគោលការណ៍ ("ជាសីឡើងរយៈពេលត្រឡប់មកវិញរបស់អ្នកមានរយៈពេលប៉ុន្មាន?") គួរត្រូវបានឆ្លើយតបពីប្រភពមានស្មារតីទុកចិត្ត មិនមែនពីអនុស្សាវរីយ៍នៃម៉ូដែល។ យើងដាក់មូលដ្ឋានចំណេះដឹងតូចមួយជាសម្លេងស្វែងរក។

នៅក្នុងផលិតកម្មនេះគឺជាការស្វែងរក Azure AI; នៅទីនេះ យើងផ្ដល់ជាស្វែងរកពាក្យគន្លឹះក្នុងអង្គចងចាំ ដើម្បីឲ្យសៀវភៅកំណត់ហេតុនេះអាចដំណើរការនៅកន្លែងណារបស់ពិភពលោក ហើយផ្លាស់ប្តូរទៅ Azure AI Search ជាអូតូម៉ាទិចពេលវាលបរិស្ថានមានវត្តមាន។


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. មេម៉ូរី

អ្នកគាំទ្រម្នាក់ដែលភ្លេចថា គាត់កំពុងនិយាយជាមួយនរណាគឺជាអ្នកគាំទ្រដែលអាក្រក់។ យើងរក្សាទុកប្រវត្តិរូបតូចមួយសម្រាប់អតិថិជននីមួយៗ ហើយបញ្ចូលសេចក្ដីសង្ខេបខ្លីទៅកាន់សេចក្ដីណែនាំរបស់អ្នកគាំទ្រ។ នៅក្នុងការផលិត វាគឺជាសេវាមេម៉ូរី (សូមមើលមេរៀនទី ១៣) ហើយនៅទីនេះ dict បានធ្វើឲ្យគំរូនេះច្បាស់លាស់។


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. ការជំនួសម៉ូដែល និងការផ្ទុកចម្លើយជាមុន

មានឧបករណ៍ថ្លៃពីរ ត្រូវបានភ្ជាប់ទៅកាន់អ្នកដោះស្រាយសំណើតែមួយៈ

- **ការជំនួស**៖ ជម្រាប់គំនិតថ្លៃទាបមួយសម្រេចថាតើសំណើនេះត្រូវបានប្រើម៉ូដែលតូច ឬ ម៉ូដែលធំ។
- **ការផ្ទុក**៖ សំនួរដែលធម្មតាប្រសើរឡើងត្រូវបានបម្រើដោយផ្ទាល់ពីកាសែចដោយគ្មានការហៅម៉ូដែល។

ជម្រាប់នៅទីនេះគឺសាមញ្ញយ៉ាងច្បាស់។ នៅក្នុងការផលិត អ្នកនឹងផ្ទៀងផ្ទាត់វាជាមួយចរាចរយ៉ាងហោចណាស់ ហើយអាចប្ដូរវាជាមួយ Model Router របស់ Foundry។


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. តំណាង​ ការយល់ព្រមរបស់មនុស្ស និងភាពអាចមើលឃើញ

ឥឡូវនេះយើងប្រមូលបង្កើតតំណាងពីឧបករណ៍ខាងលើ ហើយបម្លែងការស្នើសុំម្ដងម្កាលទាំងអស់ជាក្នុងស្ពាន OpenTelemetry។ អនុគមន៍ `handle_support_request` គឺជាអ្នកកាន់ក្នងសំណើផលិតកម្ម៖ ចងចាំ → ផ្លូវការ → តាមដាន → រត់ → ចងចាំ។

ការយល់ព្រមរបស់មនុស្សត្រូវបានគ្រប់គ្រងដោយស៊ុមផ្គុំ៖ ព្រោះ `issue_refund` មាន `approval_mode="always_require"` ការរត់នឹងឈប់បណ្តោះអាសន្ន ហើយបង្ហាញសំណើការយល់ព្រមដែលយើងដោះស្រាយយ៉ាងច្បាស់លាស់។


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. ទ្វារវាយតម្លៃ

នេះគឺជាទ្វារចេញពីមេរៀន៖ សំណុំតេស្តប្រតិបត្តិនៅក្រៅបណ្ដាញវាយតម្លៃភ្នាក់ងារហើយការបង្ហោះអនុវត្តន៍តែប៉ុណ្ណោះ ប្រសិនបើអត្រាចោលឆ្លងកាត់លើសកម្រិតមួយ។ អ្នកវាយតម្លៃនៅទីនេះគឺជាការត្រួតពិនិត្យលើការចាក់ស្វ័យគន្លឹះគ្រាន់ៗ ដើម្បីរក្សាឲ្យកំណត់ត្រាគឺជាឯកត្តា; នៅក្នុងការផលិតអ្នកនឹងប្រើ LLM ជាសលោកម៉នឬអ្នកវាយតម្លៃស្រុបទ្រង់ (មើលមេរៀនទី ១០)។


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ការចងក្រងគ្នា៖ ការចេញផ្សាយដែលបានសំណើត្រឹមត្រូវ

សែលខាងក្រោមបង្ហាញពីចំណុចទាំងមូលដែលមេរៀនបានពិពណ៌នា៖ បើករង្វិលវាយតម្លៃ ហើយ "ចេញផ្សាយ" តែប៉ុណ្ណោះ ប្រសិនបើវាឆ្លងកាត់។ នេះជាគំរូដែលអ្នកនឹងរត់នៅក្នុង CI មុនពេលផ្សព្វផ្សាយជាពាណិជ្ជកម្មជំនាន់ភ្នាក់ងារទៅសេវាកម្មអាជីព Foundry Agent ។


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## សង្ខេប

អ្នកបានប្រមួលភ្នាក់ងារគាំទ្ររបស់អតិថិជនដែលរួចរាល់សម្រាប់ផលិតកម្មជាមួយនឹងបញ្ហាប្រតិបត្តិការទាំងអស់ដែលបានភ្ជាប់:

- **ឧបករណ៍, RAG, និងការចងចាំ** ផ្តល់សមត្ថភាពនិងបរិបទឲ្យភ្នាក់ងារ។
- **ការចែកចាយម៉ូដែល និងការផ្ទុក cache** រក្សា latency និងការចំណាយឲ្យនៅក្រោមការគ្រប់គ្រង។
- **ការអនុម័តដោយមនុស្ស** គ្រប់គ្រងសកម្មភាពដែលមានហានិភ័យខ្ពស់ដូចជា ការបង្វិលប្រាក់ច្រើន។
- **ច្រកវាយតម្លៃ** បារបការចេញផ្សាយដដែលមួយមុនពេលវាផ្ញើចេញ។
- **ការតាមដាន** ធ្វើឲ្យគ្រប់សំណើអាចមើលឃើញបាន។

### ការប្រឈម

ពង្រីកភ្នាក់ងារនេះទៅជា:

1. **គាំទ្រ ម៉ូដែល ច្រើន** — បន្ថែមជាន់ទីបី "ការសន្និដ្ឋាន" ហើយបញ្ជូនការឡើងកម្ពស់/បណ្តឹងទៅវា។
2. **បន្ថែមច្រកវាយតម្លៃ** — ពង្រីក `TEST_CASES` ដើម្បីរួមបញ្ចូលសេណារីយ៉ូអនុម័តការបង្វិលប្រាក់ ហើយបញ្ជាក់ច្រកអាចចាប់ការឡើងវិញ។
3. **បន្ថែមការចែកចាយដែលយកចំណាយចិត្តទុកដាក់** — តាមដានថ្លៃដែលគេគិតថាបានសម្រាប់គ្រប់សំណើ (តិច ប្រாயមធំ ឬ cache) ហើយបោះពុម្ពរបាយការណ៍ថ្លៃក្រោយពីប៉ុស្តិកសំណួរលាយបញ្ចូលគ្នា។

នៅមេរៀនបន្ទាប់ អ្នកនឹងធ្វើដំណើរផ្ទុយគ្នា ហើយរត់ភ្នាក់ងារមួយទាំងស្រុងលើម៉ាស៊ីនផ្ទាល់ខ្លួនរបស់អ្នកជាមួយ Microsoft Foundry Local និង Qwen។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
